<div style="background-color: #0f3460; background: linear-gradient(135deg, #0f3460, #533483, #e94560); padding: 30px 30px 20px 30px; border-radius: 12px; margin-bottom: 20px;">
  <h1 style="color: #ffffff; margin: 0; font-size: 2em;">🔍 Lab 04: Trace Your Travel Agent</h1>
  <p style="color: #cccccc; font-size: 1.1em; margin-top: 8px;">
    Contoso Travel Workshop — Microsoft Foundry Agent Observability
  </p>
</div>

## 🎯 私たちが学ぶこと

このラボでは、Contoso Travelエージェントに**OpenTelemetryトレース**を組み込みます。まずローカルデバッグのために**コンソール**に、次に本番環境の可観測性を確保するために**Azure Monitor**にトレースを送信します。エージェント内部で何が起こっているかを正確に確認できます。ユーザーのクエリ、ツールの呼び出し、モデルの呼び出し、応答の生成などです。

> **トレーシングをエージェントのフライトレコーダーと考えてください。** 航空機のブラックボックスがすべての計器の読み取り値と操縦士の操作を記録するのと同様に、OpenTelemetryトレーシングはすべての操作を記録します。問題が発生した場合、何が起こったのかを正確に再現し、原因を特定することができます。

---


## 🧳 Contoso Travel のストーリー

Lab 03b の Contoso Travel マルチエージェントシステムは、開発環境では順調に動作しています。フライトエージェントは実際のフライトを見つけ、ホテルエージェントは実際の宿泊施設を推薦し、カーレンタルエージェントはライブ在庫から情報を取得します。チームは興奮しています — しかし、最初のバグ報告が届くまでです。テスターは次のように報告します: *"パリで予算内のホテルを探したのに、$450/泊の高級ホテルを推薦されました。エージェントはそれを「手頃」と言いました。"*

### 🔴 現在直面している問題

- **見えないものはデバッグできません。** エージェントシステムが複雑化するにつれ — 複数のエージェント、ツール呼び出し、ハンドオフ — 何が問題だったのかを理解することはほぼ不可能になります。
- すべてのステップを確認する必要があります: どのエージェントが呼び出されたか、どのツールが使用されたか、各ステップにかかった時間、コンポーネント間でどのデータが流れたか。
- この可視性がなければ、すべてのバグは推測ゲームになります。

### ✅ このラボで解決すること

このラボでは、旅行エージェントに**OpenTelemetryトレーシング**を追加します:
 - **コンソール**へのトレース出力によるローカルデバッグ — ノートブック内でスパン、期間、属性を確認できます。
 - **Azure Monitor**への接続により、トレースがFoundryポータルのTracingタブに流れ、本番環境での可観測性を確保します。

ラボの最後には、問題が発生した場合、どこで何が起こったのかを正確に把握できるようになります。

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part A: Console Tracing</h2>
</div>

## 2. Setup

---


In [ ]:
# 環境変数を読み込み、GenAIトレースフラグを有効にする
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.path.abspath(".")), ".env"))

# トレースモジュールをインポートする前に設定する必要があります。
# Azure SDK の実験的な GenAI スパン生成を有効にします。
os.environ["AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING"] = "true"
# トレースにプロンプト/応答の内容を完全にキャプチャします。
# PIIを含む場合は本番環境では無効にしてください。
os.environ["OTEL_INSTRUMENTATION_GENAI_CAPTURE_MESSAGE_CONTENT"] = "true"

print("✅ Environment variables configured for GenAI tracing")


In [ ]:
# OpenTelemetry を構成して Azure SDK の呼び出しをトレースする
from azure.core.settings import settings
# Azure SDK に HTTP 呼び出しの OTel スパンを発行させる
settings.tracing_implementation = "opentelemetry"

from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, ConsoleSpanExporter
from azure.ai.projects.telemetry import AIProjectInstrumentor

# ConsoleSpanExporter はスパンを標準出力に出力します (開発環境のみ)
span_exporter = ConsoleSpanExporter()
# TracerProvider はスパンの作成とライフサイクルを管理します
tracer_provider = TracerProvider()
# SimpleSpanProcessor は各スパンを同期的にエクスポートします;
# 本番環境ではパフォーマンスのために BatchSpanProcessor を使用してください。
tracer_provider.add_span_processor(SimpleSpanProcessor(span_exporter))
trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer("contoso-travel-tracing")

# Azure AI SDK を自動的に計測: エージェントの呼び出し、ツールの
# 呼び出し、モデルのリクエストが自動的にスパンを生成します。
AIProjectInstrumentor().instrument()

print("✅ Console tracing configured")
print("   Traces will appear in the output below each cell")

In [ ]:
# Microsoft Foundry に接続します (ここでは allow_preview は不要です。トレースは GA 機能で動作します)。
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
model_name = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Connected to Microsoft Foundry")

## 3. Travel Agent の作成とトレース

---


In [ ]:
# 簡単なトレース付きエージェントを作成して可観測性デモを行う
agent = project_client.agents.create_version(
    agent_name="contoso-travel-traced",
    definition=PromptAgentDefinition(
        model=model_name,
        instructions="あなたは Contoso Travel のコンシェルジュです。旅行先、フライト、ホテル、レンタカーなど、お客様からの旅行に関するご質問にお答えください。",
    ),
)
print(f"✅ エージェントが作成されました: {agent.name} (v{agent.version})")

## 4. トレース付き travel-query の実行

このセルを実行し、以下のコンソール出力を確認してください。各操作に対してOpenTelemetryスパンが表示されます。
---


In [ ]:
# start_as_current_span は親スパンを作成します。
# 内部のすべての SDK 呼び出しは、同じトレース内の子スパンになります。
with tracer.start_as_current_span("contoso-travel-query"):
    conversation = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="ヨーロッパで夏休みを過ごすのに最適な旅行先はどこですか？",
    )

    print(f"\n🤖 エージェントの応答:\n{response.output_text}")
# 上記のコンソール出力で trace_id、span_id、
# parent_id、およびトークン数などの属性を確認してください。

## 5. トレース出力の解釈

上記のコンソール出力には、以下のようなトレーススパンが含まれています。

- **`contoso-travel-query`** — 作成した親スパン
  - **`create_conversation`** — 会話の作成
  - **`invoke_agent`** — モデル推論の呼び出し

各スパンには以下が含まれます:
- `trace_id` — 1つのトレース内のすべてのスパンをグループ化
- `span_id` — このスパンの一意のID
- `parent_id` — 子スパンを親にリンク
- `attributes` — モデル名、トークン数などのメタデータ

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #2196f3; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>💡 Tip:</strong> 複数のエージェントが関与するワークフローでは、各エージェントの呼び出しに対してネストされたスパンが表示され、実行パス全体を簡単に追跡できます。</div>

---


<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #533483; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 20px 0;">
  <h2 style="margin: 0;">Part B: Azure Monitor トレース</h2>
</div>

## 6. Azure Monitor トレースの設定

---


In [ ]:
# 重複するスパンを避けるためにコンソールトレーサーをシャットダウンします
tracer_provider.shutdown()

from azure.monitor.opentelemetry import configure_azure_monitor

# このFoundryプロジェクトにリンクされているApp Insightsの接続文字列を取得します。
# 手動設定は不要です。
app_insights_connection_string = project_client.telemetry.get_application_insights_connection_string()

# console exporter をAzure Monitor exporter に置き換えます。
# 新しいTracerProviderとBatchSpanProcessorを設定し、スパンをApplication Insightsに自動的に送信します。
configure_azure_monitor(connection_string=app_insights_connection_string)

# プロバイダを再設定した後、新しいトレーサーを取得する必要があります
tracer = trace.get_tracer("contoso-travel-tracing")

print("✅ Azure Monitor トレースが設定されました")
print(f"   トレースは Application Insights に送信されます")
print(f"   Foundry ポータルの → トレース タブで確認できます")

## 7. トレース付き travel-query の実行 (Azure Monitor)

このトレースは Foundry ポータルに送信されます。トレースが Azure Monitor に表示されるまでに数分かかる場合があります。

---


In [ ]:
# Azure Monitor exporter を使用してトレース付きクエリを実行
with tracer.start_as_current_span("contoso-travel-paris-query") as span:
    # カスタム属性を設定すると、Azure Monitorでのフィルタリングが可能になります。
    # 一貫した名前空間（例: "travel.", "contoso."）を使用すると、
    # 目的地やエージェントごとにトレースをクエリできます。
    span.set_attribute("travel.destination", "Paris")
    span.set_attribute("travel.query_type", "trip_planning")
    span.set_attribute("contoso.agent_name", agent.name)

    conversation2 = openai_client.conversations.create()

    response = openai_client.responses.create(
        conversation=conversation2.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="パリへ1週間の旅行を計画しています。費用、天気、必見の観光スポットについて知っておくべきことは何ですか？",
    )

    print(f"🤖 Agent Response:\n{response.output_text}")
    print(f"\n📊 トレースは Azure Monitor に送信されました！")
    # trace_id in hex — use this to find the trace in portal
    print(f"   Trace ID: {span.get_span_context().trace_id:032x}")

## 8. Foundry ポータルでトレースを表示

トレースを表示するには:

1. [Microsoft Foundry ポータル](https://ai.azure.com) にアクセスします
2. プロジェクトを開きます
3. エージェントの **Tracing** タブをクリックします
4. 定義したスパン名でトレースが一覧表示されます
5. Trac eをクリックして、完全なスパントリーを確認します

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 4px solid #9c27b0; padding: 10px 15px; border-radius: 0 6px 6px 0; margin: 10px 0;"><strong>⏱️ Note:</strong>実行後、Azure Monitorにトレースが表示されるまで2～5分かかる場合があります。</div>


---


## 9. Custom Span Attributes

カスタム属性を使用することで、トレースのフィルタリングや分析がより効果的になります。ビジネスレベルのクエリを可能にするために、旅行固有のメタデータを追加してください。

---


In [ ]:
# ビジネスレベルのフィルタリングのための、より豊富なカスタム属性。
# これらは Azure Monitor の customDimensions に表示され、App Insights で KQL を使用してクエリを実行できます。
with tracer.start_as_current_span("contoso-travel-london-query") as span:
    span.set_attribute("travel.destination", "London")
    span.set_attribute("travel.query_type", "budget_inquiry")
    span.set_attribute("travel.customer_segment", "business")
    span.set_attribute("contoso.agent_name", agent.name)
    span.set_attribute("contoso.agent_version", str(agent.version))

    response = openai_client.responses.create(
        conversation=conversation2.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="ニューヨークからロンドンへのビジネスクラスの旅行オプションにはどのようなものがありますか？",
    )

    print(f"🤖 Agent Response:\n{response.output_text}")
    print(f"\n📊 Custom attributes added to trace:")
    print(f"   travel.destination = London")
    print(f"   travel.query_type = budget_inquiry")
    print(f"   travel.customer_segment = business")

## 10. Cleanup

---


In [ ]:
# クリーンアップ: リモート リソースを削除し、クライアントを閉じます
openai_client.conversations.delete(conversation_id=conversation.id)
openai_client.conversations.delete(conversation_id=conversation2.id)
print("✅ 会話が削除されました")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("✅ エージェントが削除されました")

openai_client.close()
project_client.close()
credential.close()
print("✅ クライアントが閉じられました")

## 11. Summary & Next Steps

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #2e8b57; padding: 18px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <h3 style="margin-top: 0; color: #1a1a1a;">✅ 達成したこと</h3>
  <ul style="margin-bottom: 0;">
  <li>ローカルデバッグ用に <b>コンソール トレース</b> を <code>ConsoleSpanExporter</code> と <code>AIProjectInstrumentor</code> で構成しました</li>
  <li>本番環境の可観測性のために <b>Azure Monitor トレース</b> を <code>configure_azure_monitor</code> で構成しました</li>
  <li>トレース付きの旅行クエリを実行し、スパン出力を確認しました</li>
  </ul>
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #1565c0; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>💡 重要なポイント:</strong> トレースはエージェントの実行を可視化するX線のようなものです。コンソールトレースは開発に最適であり、Azure Monitorトレースは本番環境の監視に不可欠です。カスタム属性を使用すると、ビジネスに関連する次元（例: 目的地、顧客セグメント）でトレースをフィルタリングできます。
</div>

<div style="background: #f0f0f0; color: #1a1a1a; border-left: 5px solid #ff9800; padding: 15px 20px; border-radius: 0 8px 8px 0; margin: 15px 0;">
  <strong>➡️ 次のステップ:</strong> <a href="lab-05-evaluation.ipynb">Lab 05</a> では、Contoso Travel エージェントの品質と安全性を評価します — 正確で流暢かつ安全な旅行アドバイスを提供することを確認します！
</div>

---
